 # Actividad 1 — Análisis exploratorio EDA

**Técnicas de Análisis Estadístico de Modelos Supervisados**
**Datos:** *Mice Protein Expression* (UCI Machine Learning Repository, id 342). Niveles de expresión de 77 proteínas
en corteza cerebral de ratones control y trisómicos (modelo de síndrome de Down), con y sin memantina, con y sin
estimulación de aprendizaje (Higuera et al., 2015).

**Objetivo del notebook:** inspeccionar, describir y preparar la base para las actividades 2 (regresión) y 3 (clasificación).

| Sección | Contenido |
|---|---|
| 0 | Preparación del entorno |
| 1 | Carga, estructura y tamaño efectivo |
| 2 | Análisis descriptivo |
| 3 | Valores faltantes |
| 4 | Valores atípicos | 
| 5 | Relaciones entre variables |
| 6 | Base depurada |



## 0. Preparación del entorno

In [19]:
# Librerias externas
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns

# Paquete del proyecto: semilla, rutas y funciones de carga
from mice_supervisado.config import SEMILLA, ruta_figuras
from mice_supervisado.datos import cargar_crudos

# Configuración de la sesión 
np.random.seed(SEMILLA)
pd.set_option("display.max_columns", 90)
FIGURAS = ruta_figuras(1)

## 1. Carga de datos y primeras observaciones

Esta sección responde tres preguntas antes de cualquier análisis: qué contiene la base, qué representa cada fila
y cuántas unidades independientes hay realmente.

In [20]:
# Lectura del archivo original de UCI sin transformaciones
df = cargar_crudos()

### 1.1 Dimensiones y tipos

In [21]:
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas\n")

# Conteo de columnas por tipo (resumen de las 82 columnas)
print(df.dtypes.value_counts())

Dimensiones: 1080 filas x 82 columnas

float64    77
str         5
Name: count, dtype: int64


### 1.2 Clasificación de columnas

In [4]:
# Las proteínas comparten el sufijo "_N" (nivel de expresión nuclear)
proteinas = [col for col in df.columns if col.endswith("_N")]
diseno = ["Genotype", "Treatment", "Behavior", "class"]
identificador = ["MouseID"]

# Verificación: las tres listas deben cubrir todas las columnas sin sobrantes
assert len(proteinas) + len(diseno) + len(identificador) == df.shape[1]
print(f"Proteínas: {len(proteinas)} | Diseño: {len(diseno)} | Identificador: {len(identificador)}")

Proteínas: 77 | Diseño: 4 | Identificador: 1


### 1.3 Valores faltantes

El análisis detallado se desarrolla en la sección 3

In [23]:
# Verificación: las variables de diseño y el identificador no tienen faltantes
assert df[diseno + identificador].notna().all().all()

# Faltantes en proteínas: propoción de celdas y de filas afectadas
total_celdas =df[proteinas].size
total_na = df[proteinas].isna().sum().sum()
filas_con_na = df[proteinas].isna().any(axis=1).sum()

print(f"Celdas faltantes: {total_na} de {total_celdas} ({total_na / total_celdas:.1%})")
print(f"Filas con al menos un faltante: {filas_con_na} de {len(df)} ({filas_con_na / len(df):.1%})")

Celdas faltantes: 1396 de 83160 (1.7%)
Filas con al menos un faltante: 528 de 1080 (48.9%)


### 1.4 Unicidad y duplicados 

In [29]:
# MouseID debe identificar cada fila de forma única
print(f"¿MouseID único por fila? {df["MouseID"].is_unique}")

# Filas con valores de proteínas idénticos, aunque tengan distinto ID
print(f"Filas con proteínas idénticas: {df.duplicated(subset=proteinas).sum()}")

# Columnas de proteínas con valores idénticos en todas las filas (posible duplicado de variable)
columnas_duplicadas = df[proteinas].T.duplicated(keep=False)
print(f"Proteínas con una columna gemela: {list(columnas_duplicadas[columnas_duplicadas].index)}")

¿MouseID único por fila? True
Filas con proteínas idénticas: 0
Proteínas con una columna gemela: ['ARC_N', 'pS6_N']


### 1.5 Columnas raton y replica

`MouseID` tiene la forma `<ratón>_<réplica>`: cada ratón se midió varias veces. Separar ambas partes permite
identificar al individuo, que es la unidad independiente del estudio.

In [30]:
# rsplit con n=1 corta solo en el último "_", de modo que la réplica siempre queda a la derecha
df[["raton", "replica"]] = df["MouseID"].str.rsplit("_", n=1, expand=True)
df["replica"] = df["replica"].astype(int)

df[["MouseID", "raton", "replica"]].head()

,MouseID,raton,replica
0,309_1,309,1
1,309_2,309,2
2,309_3,309,3
3,309_4,309,4
4,309_5,309,5


### 1.6 Verificación de raton

In [31]:
# Cada ratón pertenece a una sola clase; de lo contrario "raton" no identificaría individuos
clases_por_raton = df.groupby("raton")["class"].nunique()
print(f"Ratones asignados a más de una clase: {(clases_por_raton > 1).sum()}")

Ratones asignados a más de una clase: 0


### 1.7 Mediciones por ratón

In [32]:
mediciones = df["raton"].value_counts()

print(f"Ratones distintos: {df['raton'].nunique()}")
print(f"Mediciones por ratón: mÍn {mediciones.min()}, máx {mediciones.max()}")

# Distribución: cuántos ratones tienen cada número de mediciones
print(mediciones.value_counts().rename("n_ratones"))

Ratones distintos: 72
Mediciones por ratón: mÍn 15, máx 15
count
15    72
Name: n_ratones, dtype: int64


### 1.8 Tamaño efectivo por clase

El tamaño efectivo es el número de ratones, no el de filas: las réplicas de un mismo ratón no son observaciones
independientes (ver sección 2.3).

In [33]:
# Ratones distintos (tamaño efectivo) frente a filas (mediciones) por clase
resumen_clases = (
    df.groupby("class")
    .agg(ratones=("raton", "nunique"), filas=("MouseID", "size"))
    .sort_index()
)

resumen_clases.loc["Total"] = resumen_clases.sum()
resumen_clases

,ratones,filas
class,,
c-CS-m,10,150
c-CS-s,9,135
c-SC-m,10,150
c-SC-s,9,135
t-CS-m,9,135
t-CS-s,7,105
t-SC-m,9,135
t-SC-s,9,135
Total,72,1080


## 2. Análisis descriptivo

### 2.1 Medidas de tendencia central, dispersión y forma

Se usa el coeficiente de variación (CV = desviación estándar / media) para comparar la variabilidad entre proteínas
con escalas distintas. Es aplicable porque los niveles de expresión son positivos y tienen cero absoluto, condición
necesaria para que el CV tenga sentido (Abdi, 2010).

In [35]:
def rango(x):
    """Rango: máximo menos mínimo"""
    return x.max() - x.min()

def iqr(x):
    """Rango intercuartílico: percentil 75 menos percentil 25"""
    return x.quantile(0.75) - x.quantile(0.25)

def cv(x):
    """Coeficiente de variación: desviación estándar relativa a la media"""
    return x.std() / x.mean()

# Una fila por proteína; los NaN se ignoran, por eso "count" varía entre proteínas
# "kurt" es el exceso de curtosis (distribución normal = 0)
resumen = (
    df[proteinas].agg(
    ["count", "mean", "median", "std", "min", "max", rango, iqr, "skew", "kurt", cv])
    .T
)

print("Proteínas con mayor coeficiente de variación")
display(resumen.nlargest(10, "cv").round(3))

print("Proteínas con menor coeficiente de variación")
display(resumen.nsmallest(10, "cv").round(3))

Proteínas con mayor coeficiente de variación


,count,mean,median,std,min,max,rango,iqr,skew,kurt,cv
AcetylH3K9_N,1080.0,0.216,0.150,0.185,0.053,1.459,1.407,0.166,2.682,8.964,0.856
pERK_N,1077.0,0.546,0.444,0.345,0.149,3.567,3.418,0.326,3.835,22.407,0.633
DYRK1A_N,1077.0,0.426,0.366,0.249,0.145,2.516,2.371,0.200,4.357,27.351,0.586
BRAF_N,1077.0,0.379,0.327,0.216,0.144,2.133,1.990,0.149,4.548,28.226,0.572
SOD1_N,1077.0,0.543,0.444,0.281,0.217,1.873,1.656,0.376,1.301,1.683,0.517
ITSN1_N,1077.0,0.617,0.566,0.252,0.245,2.603,2.357,0.225,3.622,21.518,0.408
pP70S6_N,1077.0,0.394,0.378,0.156,0.131,1.129,0.998,0.200,0.900,1.713,0.396
pCAMKII_N,1077.0,3.537,3.327,1.295,1.344,7.464,6.120,2.002,0.548,-0.579,0.366
H3AcK18_N,900.0,0.170,0.158,0.059,0.080,0.480,0.400,0.072,1.757,4.915,0.350
pPKCG_N,1080.0,1.707,1.665,0.578,0.599,3.382,2.783,0.816,0.251,-0.501,0.339


Proteínas con menor coeficiente de variación


,count,mean,median,std,min,max,rango,iqr,skew,kurt,cv
ERBB4_N,1080.0,0.156,0.156,0.015,0.100,0.209,0.108,0.018,-0.031,0.580,0.096
P3525_N,1080.0,0.291,0.291,0.030,0.207,0.444,0.236,0.042,0.184,0.139,0.103
BAX_N,1080.0,0.179,0.181,0.019,0.072,0.241,0.169,0.023,-0.591,1.406,0.105
GFAP_N,1080.0,0.121,0.120,0.013,0.086,0.214,0.128,0.015,0.862,3.952,0.109
pGSK3B_Tyr216_N,1080.0,0.849,0.850,0.094,0.577,1.205,0.627,0.122,-0.109,0.050,0.111
PSD95_N,1080.0,2.235,2.242,0.254,1.206,2.878,1.672,0.341,-0.454,0.484,0.114
ARC_N,1080.0,0.122,0.122,0.014,0.067,0.159,0.091,0.021,-0.078,-0.381,0.117
pS6_N,1080.0,0.122,0.122,0.014,0.067,0.159,0.091,0.021,-0.078,-0.381,0.117
pGSK3B_N,1080.0,0.161,0.160,0.019,0.100,0.253,0.153,0.022,0.543,1.348,0.120
CDK5_N,1080.0,0.292,0.294,0.037,0.181,0.817,0.636,0.040,2.485,35.446,0.128


### 2.2 Forma de las distribuciones

Criterio: |asimetría| < 0,5 se considera aproximadamente simétrica, entre 0,5 y 1 moderada, y > 1 fuerte
(Bulmer, 1979). En este análisis se reportan las proteínas con asimetría fuerte.

In [36]:
UMBRAL_ASIMETRIA = 1

fuerte = resumen["skew"].abs() > UMBRAL_ASIMETRIA
derecha = resumen["skew"] > UMBRAL_ASIMETRIA
izquierda = resumen["skew"] < -UMBRAL_ASIMETRIA

print(f"Asimetría fuerte: {fuerte.sum()} de {len(resumen)} proteínas")
print(f"  a la derecha: {derecha.sum()} | a la izquierda: {izquierda.sum()}")

# Detalle de las proteínas con asimetría fuerte, de mayor a menor
resumen.loc[fuerte, ["skew", "kurt", "cv"]].sort_values("skew", ascending=False).round(3)

Asimetría fuerte: 17 de 77 proteínas
  a la derecha: 17 | a la izquierda: 0


,skew,kurt,cv
GluR4_N,4.739,62.549,0.212
BRAF_N,4.548,28.226,0.572
DYRK1A_N,4.357,27.351,0.586
RRP1_N,4.004,46.118,0.191
pELK_N,3.995,25.655,0.327
pERK_N,3.835,22.407,0.633
ITSN1_N,3.622,21.518,0.408
AcetylH3K9_N,2.682,8.964,0.856
CDK5_N,2.485,35.446,0.128
Tau_N,1.867,4.575,0.328


### 2.3 Variabilidad dentro de cada ratón frente a variabilidad entre ratones

Para cada proteína se comparan dos dispersiones:

- **dentro:** desviación estándar de las réplicas de cada ratón, promediada sobre los 72 ratones (ruido de medición).
- **entre:** desviación estándar de las medias de los 72 ratones (diferencias entre individuos y clases).

**Valor de referencia.** Si no hubiera diferencias reales entre ratones, la media de cada ratón variaría solo por
el ruido de sus *n* réplicas. El error estándar de una media muestral es σ/√n (James et al., 2023), de modo que con
*n* = 15 la razón entre/dentro esperada sería ≈ 1/√15 ≈ 0,26. Una razón claramente mayor que ese valor indica un
efecto de ratón; una razón mayor que 1 indica un efecto fuerte.

In [37]:
por_raton = df.groupby("raton")[proteinas]

variabilidad = pd.DataFrame({
    "dentro": por_raton.std().mean(),
    "entre": por_raton.mean().std(),
})
variabilidad["razon"] = variabilidad["entre"] / variabilidad["dentro"]

# Razón esperada si toda la variación fuera ruido de medición (sin efecto de ratón)
referencia = 1 / np.sqrt(mediciones.median())

efecto_fuerte = variabilidad["razon"] > 1
efecto_moderado = variabilidad["razon"].between(referencia, 1, inclusive="right")
sin_efecto = variabilidad["razon"] <= referencia

print(f"Razón de referencia sin efecto de ratón: {referencia:.2f}")
print(f"Mediana de la razón: {variabilidad['razon'].median():.2f}")
print(f"Efecto de ratón fuerte (razón > 1): {efecto_fuerte.sum()} de {len(variabilidad)}")
print(f"Efecto moderado ({referencia:.2f} < razón ≤ 1): {efecto_moderado.sum()}")
print(f"Sin efecto aparente (razón ≤ {referencia:.2f}): {sin_efecto.sum()}")

# Proteínas con razón ≤ 1, de menor a mayor
variabilidad.loc[~efecto_fuerte].sort_values("razon").round(3)

Razón de referencia sin efecto de ratón: 0.26
Mediana de la razón: 1.74
Efecto de ratón fuerte (razón > 1): 66 de 77
Efecto moderado (0.26 < razón ≤ 1): 11
Sin efecto aparente (razón ≤ 0.26): 0


,dentro,entre,razon
ELK_N,0.265,0.184,0.696
Bcatenin_N,0.351,0.261,0.743
pCFOS_N,0.017,0.015,0.887
BAD_N,0.021,0.019,0.900
P70S6_N,0.123,0.115,0.941
pNR1_N,0.085,0.080,0.947
NR2A_N,0.676,0.643,0.950
BAX_N,0.013,0.013,0.958
NUMB_N,0.021,0.020,0.974
pNR2B_N,0.191,0.187,0.980
